# Alert 생성기 실험 ⑦ — 일반 룰(Oracle 시나리오) 4종의 기본값 고르기

룰엔진의 일반 템플릿 4종을 03번과 **같은 원장 · 같은 구간 · 같은 지표**로 돌려, 템플릿에 넣을 기본값을 고른다. 모델도 생성기도 쓰지 않는다.

| 룰 | Oracle 시나리오 | 한 줄 정의 | 대상 |
|---|---|---|---|
| 단건 고액거래 | Large Reportable Transactions | 거래 1건의 금액이 기준 이상 | 거래 |
| 분할 입 · 출금 | Structuring – Avoidance of Reporting Thresholds | 기간 안에 보고 기준 바로 아래 금액의 거래를 N 건 이상 | 계좌 |
| 같은 금액 반복 | Deposits/Withdrawals in Same or Similar Amounts | 기간 안에 검사 거래와 같은(±오차) 금액의 거래를 N 건 이상 | 계좌 |
| 급속 자금 통과 | Rapid Movement of Funds – All Activity | 기간 안에 받은 합계가 기준 이상이고, 첫 입금 뒤 내보낸 합계가 그 일정 비율 이상 | 계좌 |

- 정의는 3절과 `scenario_lib.py` 에 있다. 원장 · 거래 필터 · 바쁜 계좌는 03번의 `rule_lib.py` 를 그대로 쓴다.
- 구간, 지표에서 빼는 세탁, 바쁜 계좌 기준은 **03번 결과(`run.json`)에서 읽는다.** 03번을 같은 DATASET 으로 먼저 돌려야 한다.
- 03번과 다른 점: 패턴 룰은 룰마다 노리는 패턴이 있어서 재현율을 "그 패턴의 세탁 거래" 기준으로 쟀다. 일반 룰은 노리는 패턴이 없어서 **6종 패턴 세탁 전체**가 분모다.
  그래서 재현율은 낮게 나오는 것이 정상이고, 여기서 볼 것은 **정밀도와 양(하루 평균, 원장 대비)** 이다.
- 조건은 후보를 전부 돌린 뒤 보정 구간에서 고르고(5절) 평가 구간에서 잰다(6절). 고르는 기준은 03번과 같이 F1 이지만, 일반 룰은 F1 이 가장 큰 조건이 쓸 만한 조건이라는 보장이 없다.
  4절의 표를 보고 5절에서 직접 정하는 것을 전제로 한다.
- 룰 SQL 은 가짜 원장에서 같은 정의를 Python 반복문으로 짠 계산과 결과가 같은 것을 확인했다(2026-11-24). **실제 데이터로는 아직 안 돌려 봤다.** 소요 시간을 모른다.
  오래 걸리면 0절의 후보(특히 `PASS_FILTERS`, `SAME_FILTERS`)를 줄인다.

## 0. 설정

In [1]:
DATASET = 'HI-Medium'            # 03번을 같은 DATASET 으로 먼저 돌려야 한다
RULE_RUN = 'rule_vs_model'       # 03번 결과 폴더. 구간 · 제외 패턴 · 바쁜 계좌 기준 · 환전 금액 비를 여기서 읽는다

import json
from pathlib import Path
import gen_lib as GL

RULE_DIR = Path.cwd() / 'results' / DATASET / RULE_RUN
rule_run = json.load(open(RULE_DIR / 'run.json', encoding='utf-8'))
TEST_WINDOW, CALIB_WINDOW = tuple(rule_run['test_window']), tuple(rule_run['calib_window'])      # 평가 구간, 보정 구간
PAD_DAYS, EXCLUDE = rule_run['pad_days'], rule_run['exclude']                                    # 보정 구간 앞으로 더 읽는 날짜 수, 지표에서 빼는 세탁
BUSY_MAX, BUSY_MINUTES = rule_run['busy']                                                        # 바쁜 계좌 기준
FX_RATIO = tuple(rule_run['fx_ratio'])
print('보정', CALIB_WINDOW, '| 평가', TEST_WINDOW, '| 앞으로 더 읽는 날', PAD_DAYS, '| 지표에서 빼는 세탁', EXCLUDE, '| 바쁜 계좌', BUSY_MAX, '건 /', BUSY_MINUTES, '분')

보정 ('2022-09-05', '2022-09-07') | 평가 ('2022-09-08', '2022-09-16') | 앞으로 더 읽는 날 14 | 지표에서 빼는 세탁 ['NO_PATTERN', 'STACK', 'BIPARTITE'] | 바쁜 계좌 512 건 / 4320 분


In [2]:
# --- 거래 필터 후보 (03번과 같은 뜻). 이름: (건당 최소 금액 USD, 처음 거래하는 상대만 셀지)
FILTERS = {'없음': (0, False), '1천↑': (1000, False), '첫거래': (0, True), '첫거래+1천↑': (1000, True)}
SIDES = ['in', 'out']            # 방향을 고르는 룰(분할, 같은 금액 반복)에서 볼 방향. 'in' = 받는 계좌가 대상, 'out' = 보내는 계좌가 대상

# --- ① 단건 고액거래: 건당 금액 >= 기준
LARGE_FILTERS = ['없음', '첫거래']
LARGE_USD = [5_000, 10_000, 50_000, 100_000, 500_000, 1_000_000]

# --- ② 분할 입 · 출금: 건당 금액이 [하한, 보고 기준) 인 거래가 기간 안에 N 건 이상
REPORT_USD = 10_000              # 보고 기준. 미국 CTR 의 1만 달러를 그대로 썼다 (이 원장은 현금 구분을 쓰지 않으므로 뜻이 같지는 않다)
STRUCT_FILTERS = ['없음', '첫거래']             # 금액 조건은 띠가 대신하므로 금액 필터는 뺐다
STRUCT_LOWS = [5_000, 8_000, 9_000]            # 띠의 하한
STRUCT_WINDOWS = [1440, 4320, 10080]           # 기간(분): 1일 / 3일 / 7일
STRUCT_COUNTS = [2, 3, 4, 5, 6, 8, 10]         # 건수

# --- ③ 같은 금액 반복: 검사 거래 금액의 ±오차 안인 거래가 기간 안에 N 건 이상
SAME_FILTERS = ['없음', '1천↑', '첫거래', '첫거래+1천↑']
SAME_TOLERANCES = [0, 0.01, 0.05]              # 오차. 0 = 정확히 같은 금액
SAME_WINDOWS = [1440, 4320, 10080]
SAME_COUNTS = [2, 3, 4, 5, 6, 8, 10]

# --- ④ 급속 자금 통과: 기간 안 받은 합계 >= 최소 합계 AND 첫 입금 뒤 출금 합계 ÷ 받은 합계 >= 비율 하한 [AND <= 비율 상한]
PASS_FILTERS = ['없음', '1천↑', '첫거래+1천↑']
PASS_WINDOWS = [60, 360, 1440, 4320]           # 1시간 / 6시간 / 1일 / 3일
PASS_SUMS = [1_000, 10_000, 100_000]           # 최소 받은 합계(USD)
PASS_RATIOS = [0.5, 0.8, 0.9]                  # 출금 ÷ 입금 비율의 하한
PASS_CAPS = [False, True]                      # True = 비율 상한도 둔다(상한 = 1 ÷ 하한. 하한 0.9 면 0.9 ~ 1.11). 받은 것보다 훨씬 많이 내보내는 계좌를 뺀다

PICK_BY = 'F1'                   # 5절에서 조건을 고르는 기준 열
MIN_PICKED = 10                  # 보정 구간에서 고른 거래가 이보다 적은 후보는 고르지 않는다 (몇 건 맞혀서 정밀도 100% 가 되는 후보를 빼려고)

# --- '룰-템플릿-정의' 문서 5.3 의 가설 기본값. 6절에서 고른 값과 나란히 본다. 값은 위 후보 안에 있어야 한다
HYPOTHESIS = {
    'large_tx':     dict(필터='없음', 건당_최소=10_000),
    'structuring':  dict(방향='입금', 필터='없음', 기간_분=1440, 건당_하한=8_000, 건수=3),
    'same_amount':  dict(방향='입금', 필터='없음', 기간_분=10080, 오차=0.05, 건수=3),
    'pass_through': dict(필터='없음', 기간_분=1440, 최소_입금합계=10_000, 비율_하한=0.9, 비율_상한=None),
}

THREADS, MEMORY = 8, '24GB'      # DuckDB 설정. 서버 사양에 맞춰 고친다
FILE_DB = DATASET != 'HI-Small'  # True 면 표를 메모리가 아니라 파일(parquets/<DATASET>/scenario_work.duckdb)에 둔다. 03번의 작업 DB 와는 다른 파일이다

**후보에 대해.**
- 거래 필터는 03번에서 패턴 룰을 정확하게 만든 조건이라 일반 룰에도 걸어 본다. `첫거래` = 그 (송금 계좌 → 수취 계좌) 쌍의 첫 거래만 센다.
- 보고 기준 1만 달러 말고는 **문헌에서 가져온 숫자가 없다.** 나머지 후보(띠의 하한, 오차, 비율, 최소 합계)는 범위를 넓게 잡은 것이다. 1절의 금액 분포를 보고 고친다.
- 급속 자금 통과와 같은 금액 반복은 원장 전체를 계좌별로 돌아보는 계산이라 필터 하나에 시간이 가장 많이 든다. 먼저 필터를 하나만 두고 시간을 재 본 뒤 늘리는 것이 안전하다.

## 1. 거래 읽기
03번과 같다. 보정 구간 앞 `PAD_DAYS` 일부터 평가 구간 끝까지를 DuckDB 표로 올린다. 금액은 서비스와 같은 고정 환율로 USD 로 바꾼다.

In [3]:
import time
import duckdb, numpy as np, pandas as pd
import rule_lib as RL
import scenario_lib as SL

TAGS = {name: RL.filter_tag(*spec) for name, spec in FILTERS.items()}     # 필터 이름 → 표 이름에 붙는 꼬리표
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'parquets').is_dir()), None)
assert ROOT is not None, 'parquets/ 폴더를 못 찾음. 데이터가 있는 GPU 서버의 저장소 안에서 열어야 한다'
BASE = ROOT / 'parquets' / DATASET
OUT = Path.cwd() / 'results' / DATASET / 'scenario_rules'
OUT.mkdir(parents=True, exist_ok=True)
START = str((pd.Timestamp(CALIB_WINDOW[0]) - pd.Timedelta(days=PAD_DAYS)).date())
END = TEST_WINDOW[1]
SPANS = {'보정': CALIB_WINDOW, '평가': TEST_WINDOW}
WORK_DB = BASE / 'scenario_work.duckdb' if FILE_DB else None
con = duckdb.connect(str(WORK_DB)) if FILE_DB else duckdb.connect()
con.execute(f'SET threads={THREADS}'); con.execute(f"SET memory_limit='{MEMORY}'")
if FILE_DB:
    con.execute(f"SET temp_directory='{BASE / 'scenario_tmp'}'")     # 메모리에 안 들어가는 중간 결과를 쓰는 곳
print(BASE, '| 읽는 기간', START, '~', END, '| duckdb', duckdb.__version__, '| 작업 DB', WORK_DB)

/home/donghyeon/workspace/money_laundry_ai_/parquets/HI-Medium | 읽는 기간 2022-08-22 ~ 2022-09-16 | duckdb 2.0.0 | 작업 DB /home/donghyeon/workspace/money_laundry_ai_/parquets/HI-Medium/scenario_work.duckdb


In [4]:
RL.load_tx(con, BASE, START, END, GL.UNITS_PER_USD, EXCLUDE, FX_RATIO)     # 환율_모르는_거래 는 0 이어야 정상. 03번과 숫자가 같아야 한다

{'읽은_거래': 31891231,
 '룰이_보는_거래': 29331856,
 '시작': '2022-09-01 00:00:00',
 '끝': '2022-09-16 23:59:00',
 '환율_모르는_거래': 0,
 '지표에서_뺀_세탁': 17727,
 '환전_직후_송금': 472401}

In [5]:
# 구간별 거래 수. 재현율의 분모는 '6종 패턴 세탁' 줄이다
IN_CALIB = f"day BETWEEN DATE '{CALIB_WINDOW[0]}' AND DATE '{CALIB_WINDOW[1]}'"
IN_TEST = f"day BETWEEN DATE '{TEST_WINDOW[0]}' AND DATE '{TEST_WINDOW[1]}'"
KIND = "CASE WHEN y = 0 THEN '정상' WHEN skip THEN '지표에서 뺀 세탁' ELSE '6종 패턴 세탁' END"
con.execute(f'''SELECT {KIND} AS 구분, count(*) FILTER (WHERE {IN_CALIB}) AS 보정_거래, count(*) FILTER (WHERE {IN_TEST}) AS 평가_거래
                FROM tx GROUP BY 1 ORDER BY 1''').df()

,구분,보정_거래,평가_거래
0,6종 패턴 세탁,2533,9253
1,정상,5668852,16608620
2,지표에서 뺀 세탁,3504,10498


In [6]:
# 금액 분포 (평가 구간, 남에게 보낸 거래). 'n↑' = 건당 금액이 그 값 이상인 거래의 비율. 단건 고액거래의 기준이 거래의 몇 % 에 걸리는지 여기서 먼저 보인다
share = ', '.join(f'round(avg((usd >= {v})::INT), 4) AS "{v:,}↑"' for v in LARGE_USD)
con.execute(f'''SELECT {KIND} AS 구분, count(*) AS 거래, round(quantile_cont(usd, 0.25)) AS p25, round(median(usd)) AS 중앙값,
                       round(quantile_cont(usd, 0.75)) AS p75, round(quantile_cont(usd, 0.95)) AS p95, {share}
                FROM tx WHERE {IN_TEST} GROUP BY 1 ORDER BY 1''').df()

,구분,거래,p25,중앙값,p75,p95,"5,000↑","10,000↑","50,000↑","100,000↑","500,000↑","1,000,000↑"
0,6종 패턴 세탁,9253,5186.0,9961.0,15193.0,19764.0,0.7588,0.4982,0.0267,0.0252,0.0199,0.0187
1,정상,16608620,173.0,822.0,4254.0,114720.0,0.2288,0.1645,0.0701,0.0531,0.0248,0.0161
2,지표에서 뺀 세탁,10498,1356.0,3663.0,9617.0,32598.0,0.3661,0.2398,0.0320,0.0289,0.0221,0.0188


In [7]:
# 보고 기준 바로 아래 띠에 든 거래의 비율 (평가 구간). 분할 룰이 세는 거래가 얼마나 되는지 본다
band = ', '.join(f'round(avg((usd >= {low} AND usd < {REPORT_USD})::INT), 4) AS "{low:,} ~ {REPORT_USD:,}"' for low in STRUCT_LOWS)
con.execute(f'SELECT {KIND} AS 구분, count(*) AS 거래, {band} FROM tx WHERE {IN_TEST} GROUP BY 1 ORDER BY 1').df()

,구분,거래,"5,000 ~ 10,000","8,000 ~ 10,000","9,000 ~ 10,000"
0,6종 패턴 세탁,9253,0.2606,0.1020,0.0500
1,정상,16608620,0.0643,0.0189,0.0089
2,지표에서 뺀 세탁,10498,0.1263,0.0487,0.0256


## 2. 바쁜 계좌 · 거래 필터
03번과 같은 기준이다. 바쁜 계좌는 룰의 대상이 되지 않는다(단건 고액거래는 계좌를 돌아보지 않아서 해당 없음).

In [8]:
RL.busy_accounts(con, BUSY_MAX, BUSY_MINUTES)

{'바쁜_계좌': 15, '바쁜_계좌에_닿은_거래': 2979311, '그중_세탁': 4196}

In [9]:
pd.DataFrame([dict(필터=name, **RL.apply_filter(con, TAGS[name], *FILTERS[name])) for name in FILTERS])

,필터,tag,건당_최소_USD,첫_거래만,환전,남은_거래,그중_세탁,그중_6종_패턴_세탁
0,없음,m0a,0,False,—,29331856,31086,13359
1,1천↑,m1000a,1000,False,—,14028122,26724,12689
2,첫거래,m0f,0,True,—,2995121,22617,11085
3,첫거래+1천↑,m1000f,1000,True,—,1477553,20583,10533


## 3. 룰 돌리기
계좌를 돌아보는 세 룰(분할, 같은 금액 반복, 급속 자금 통과)은 패턴 룰과 같은 방식으로 돈다.

1. **트리거**: 대상 계좌의 거래 하나하나를 검사 시점으로 삼는다.
2. **기간**: 트리거 시각을 포함해 직전 `W` 분을 돌아본다.
3. **조건**: 그 기간 안에서 센 값이 기준을 넘으면 적중이다.
4. **근거 거래**: 적중한 트리거의 기간 안에서 그 조건을 이루는 거래. 이것이 룰이 "고른 거래"다.

각 함수는 "거래마다 어느 조건까지 근거 거래가 되는가"를 표로 저장한다. 조건값을 바꿔 가며 재는 것은 4절에서 한 번에 한다.

### 3-1. 단건 고액거래
거래 1건의 금액(USD)이 기준 이상이면 그 거래를 고른다. 기간도 대상 계좌도 없다. 표에는 가장 작은 기준 이상인 거래만 담아 둔다.

In [10]:
pd.DataFrame([dict(필터=name, **SL.large_tx(con, TAGS[name], min(LARGE_USD))) for name in LARGE_FILTERS])

,필터,표,거래,초
0,없음,large_tx_m0a,6953282,0.7
1,첫거래,large_tx_m0f,613686,0.1


### 3-2. 분할 입 · 출금

| | 입금 | 출금 |
|---|---|---|
| 대상 | 수취 계좌 A | 송금 계좌 A |
| 세는 거래 | A 로 들어온 거래 중 금액이 `[하한, REPORT_USD)` 인 것 | A 가 보낸 거래 중 금액이 그 띠 안인 것 |
| 트리거 | 띠 안 거래 하나하나 | 같음 |
| 조건 | 기간 안 띠 안 거래 수 ≥ N | 같음 |
| 근거 거래 | 기간 안 띠 안 거래 전부 | 같음 |

띠 밖의 거래는 세지도 않고 근거 거래도 되지 않는다. 상대 계좌가 같은지는 보지 않는다.

In [11]:
pd.DataFrame([dict(방향=SL.SIDES[side], 필터=name, 기간_분=W, 하한=low, **SL.structuring(con, side, W, low, REPORT_USD, TAGS[name]))
              for side in SIDES for name in STRUCT_FILTERS for W in STRUCT_WINDOWS for low in STRUCT_LOWS])

,방향,필터,기간_분,하한,표,거래,초
0,입금,없음,1440,5000,structuring_in_1440_5000_m0a,1398135,0.3
1,입금,없음,1440,8000,structuring_in_1440_8000_m0a,360209,0.1
2,입금,없음,1440,9000,structuring_in_1440_9000_m0a,161664,0.1
3,입금,없음,4320,5000,structuring_in_4320_5000_m0a,1703135,0.3
4,입금,없음,4320,8000,structuring_in_4320_8000_m0a,471382,0.1
5,입금,없음,4320,9000,structuring_in_4320_9000_m0a,218147,0.1
6,입금,없음,10080,5000,structuring_in_10080_5000_m0a,1763201,0.3
7,입금,없음,10080,8000,structuring_in_10080_8000_m0a,497354,0.2
8,입금,없음,10080,9000,structuring_in_10080_9000_m0a,231278,0.1
9,입금,첫거래,1440,5000,structuring_in_1440_5000_m0f,10417,0.1


### 3-3. 같은 금액 반복
대상 계좌의 거래 하나하나가 트리거다. 트리거 금액을 `x`, 오차를 `tol` 이라 하면

- `n` = 기간 안 같은 방향 거래 중 금액이 `x × (1 − tol)` ~ `x × (1 + tol)` 인 것의 수 (트리거 자신 포함)
- `n ≥ N` 이면 적중이고, 그 금액 범위에 든 기간 안 거래 전부가 근거 거래다
- `tol = 0` 이면 USD 환산 금액이 정확히 같은 거래만 센다

같은 계좌의 거래 쌍을 가장 긴 기간 · 가장 큰 오차로 한 번 이은 뒤, 기간 · 오차 조합별 수를 그 안에서 같이 센다. 그래서 (방향, 필터)마다 한 번씩만 돈다.
`후보_거래` 는 "같은 계좌에 금액이 비슷한 다른 거래가 있는 거래" 수다. 이 수가 크면 오래 걸린다.

In [12]:
pd.DataFrame([dict(방향=SL.SIDES[side], 필터=name, **SL.same_amount(con, side, SAME_WINDOWS, SAME_TOLERANCES, TAGS[name]))
              for side in SIDES for name in SAME_FILTERS])

,방향,필터,표,후보_거래,트리거,거래,초
0,입금,없음,same_amount_in_m0a,27553470,23564771,26024355,43.6
1,입금,1천↑,same_amount_in_m1000a,13252243,11070496,12261677,14.1
2,입금,첫거래,same_amount_in_m0f,311121,186547,294618,0.8
3,입금,첫거래+1천↑,same_amount_in_m1000f,251497,160793,246055,0.7
4,출금,없음,same_amount_out_m0a,24562233,20958707,23135273,37.0
5,출금,1천↑,same_amount_out_m1000a,11581182,9740813,10729326,13.8
6,출금,첫거래,same_amount_out_m0f,171556,65366,125843,0.5
7,출금,첫거래+1천↑,same_amount_out_m1000f,45064,18845,37116,0.2


### 3-4. 급속 자금 통과
대상은 계좌 A, 트리거는 A 의 출금이다. 트리거 시각 `t0` 에서 기간 `W` 만큼 돌아봐

- `sum_in` = 기간 안 입금 합계
- `sum_out` = **기간 안 첫 입금 시각부터** `t0` 까지 출금 합계 (받은 뒤에 내보낸 것만 센다. GATHER-SCATTER 와 같은 순서 규칙)
- 적중: `sum_in ≥ 최소 합계` AND `sum_out ÷ sum_in ≥ 비율 하한` [AND `≤ 비율 상한`]
- 근거 거래 = 기간 안 입금 전부 + 첫 입금 뒤의 출금

GATHER-SCATTER 와 뼈대가 같고, 상대 계좌 수 대신 금액 비율을 본다는 점이 다르다. 한 계좌에서 받아 한 계좌로 보내도 걸린다.
조건 조합(`PASS_COMBOS`)은 한 번 돌 때 전부 같이 계산한다. 그래서 (필터, 기간)마다 한 번씩만 돈다.

In [13]:
PASS_COMBOS = SL.pass_combos(PASS_SUMS, PASS_RATIOS, PASS_CAPS)
print('조건 조합', len(PASS_COMBOS), '벌 | 예:', PASS_COMBOS[0], PASS_COMBOS[-1])
pd.DataFrame([dict(필터=name, 기간_분=W, **SL.pass_through(con, W, TAGS[name], PASS_COMBOS)) for name in PASS_FILTERS for W in PASS_WINDOWS])

조건 조합 18 벌 | 예: {'최소_입금합계': 1000, '비율_하한': 0.5, '비율_상한': None} {'최소_입금합계': 100000, '비율_하한': 0.9, '비율_상한': 1.1111}


,필터,기간_분,표,계좌_쪽_거래,남긴_계좌,줄,초
0,없음,60,pass_through_60_m0a,55684401,95461,616982,20.9
1,없음,360,pass_through_360_m0a,55684401,193830,3399337,28.5
2,없음,1440,pass_through_1440_m0a,55684401,283333,11882428,34.4
3,없음,4320,pass_through_4320_m0a,55684401,327240,19450500,39.6
4,1천↑,60,pass_through_60_m1000a,26561275,82262,419113,10.6
5,1천↑,360,pass_through_360_m1000a,26561275,167689,2049776,14.4
6,1천↑,1440,pass_through_1440_m1000a,26561275,232001,5784637,16.5
7,1천↑,4320,pass_through_4320_m1000a,26561275,248451,7621282,17.9
8,첫거래+1천↑,60,pass_through_60_m1000f,2858877,11200,23175,1.0
9,첫거래+1천↑,360,pass_through_360_m1000f,2858877,46937,105114,1.1


## 4. 조건을 바꾸면 (보정 구간)
후보 전부를 보정 구간과 평가 구간에서 잰다. 아래 표는 룰마다 보정 구간 `PICK_BY` 가 높은 순이다.

- `대상_계좌`: 근거 거래가 걸린 서로 다른 대상 계좌 수 (단건 고액거래는 송금 계좌 수)
- `고른_거래`: 룰이 고른 거래(근거 거래) 수. 지표에서 빼는 세탁은 여기에 없다
- `정밀도`: 고른 거래 중 세탁 거래 비율 (패턴 종류 무관)
- `재현율`: **6종 패턴 세탁 전체** 중 고른 비율. 일반 룰은 노리는 패턴이 없어서 03번의 재현율보다 분모가 크다
- `하루_평균`: 고른 거래 ÷ 구간 날짜 수. `원장_대비`: 고른 거래 ÷ 구간 안 거래 전체. 이 룰을 "근거로만 표시"로 켰을 때 붙는 양이다
- `뺀_세탁`: 고른 거래 중 지표에서 뺀 세탁(무패턴 등) 수. 지표에는 안 들어가고 참고로만 본다

In [ ]:
CANDS = (SL.large_candidates(LARGE_FILTERS, TAGS, LARGE_USD)
         + SL.structuring_candidates(SIDES, STRUCT_FILTERS, TAGS, STRUCT_WINDOWS, STRUCT_LOWS, REPORT_USD, STRUCT_COUNTS)
         + SL.same_candidates(SIDES, SAME_FILTERS, TAGS, SAME_WINDOWS, SAME_TOLERANCES, SAME_COUNTS)
         + SL.pass_candidates(PASS_FILTERS, TAGS, PASS_WINDOWS, PASS_COMBOS))
measured = SL.measure(con, CANDS, SPANS)
calib = measured[measured.구간 == '보정'].reset_index(drop=True)
test = measured[measured.구간 == '평가'].reset_index(drop=True)
print(f'후보 {len(CANDS):,} 개')
calib.groupby('룰', sort=False).agg(후보=('후보', 'size'), 고른_거래_최소=('고른_거래', 'min'), 고른_거래_최대=('고른_거래', 'max'),
                                    정밀도_최대=('정밀도', 'max'), F1_최대=('F1', 'max'))

In [ ]:
SL.top(calib, 'large_tx', by=PICK_BY, min_picked=MIN_PICKED)

In [ ]:
SL.top(calib, 'structuring', by=PICK_BY, min_picked=MIN_PICKED)

In [ ]:
SL.top(calib, 'same_amount', by=PICK_BY, min_picked=MIN_PICKED)

In [ ]:
SL.top(calib, 'pass_through', by=PICK_BY, min_picked=MIN_PICKED)

In [ ]:
# 정밀도 순으로도 본다. F1 이 높은 조건은 양이 많고 정밀도가 낮은 쪽으로 쏠릴 수 있다
pd.concat({SL.LABELS[rule]: SL.top(calib, rule, n=5, by='정밀도', min_picked=MIN_PICKED) for rule in SL.LABELS})

## 5. 조건 고르기
룰마다 보정 구간에서 `PICK_BY` 가 가장 큰 후보 하나를 고른다(고른 거래가 `MIN_PICKED` 건 이상인 것 중에서).
다른 조건을 쓰려면 아래 셀에서 `CHOSEN` 을 직접 바꾼다. `SL.find` 에 조건을 적으면 그 후보의 번호를 찾아 준다.

In [ ]:
CHOSEN = SL.pick(calib, PICK_BY, MIN_PICKED)       # {룰: 후보 번호}
# 직접 정할 때 (예시)
# CHOSEN['structuring'] = SL.find(CANDS, 'structuring', 방향='입금', 필터='없음', 기간_분=1440, 건당_하한=8_000, 건수=3)
# CHOSEN['pass_through'] = SL.find(CANDS, 'pass_through', 필터='없음', 기간_분=1440, 최소_입금합계=10_000, 비율_하한=0.9, 비율_상한=None)
DROP = ['표', '식', '유일']
pd.DataFrame({SL.LABELS[rule]: {k: v for k, v in CANDS[number].items() if k not in DROP} for rule, number in CHOSEN.items()}).T.fillna('')

## 6. 평가 구간
고른 조건을 평가 구간에 그대로 건다. 여기부터는 조건을 바꾸지 않는다. 가설 기본값('룰-템플릿-정의' 문서 5.3)을 같은 구간에서 나란히 본다.

In [ ]:
SHOW = ['룰', '방향', '필터', '기간_분', '조건', '대상_계좌', '고른_거래', '세탁', '정밀도', '재현율', 'F1', '하루_평균', '원장_대비', '뺀_세탁']
GUESS = {rule: SL.find(CANDS, rule, **conditions) for rule, conditions in HYPOTHESIS.items()}
chosen_table = pd.concat([SL.chosen_rows(test, CHOSEN)[SHOW].assign(값='고른 값'), SL.chosen_rows(test, GUESS)[SHOW].assign(값='가설 기본값')])
chosen_table = chosen_table.set_index(['룰', '값']).loc[[SL.LABELS[rule] for rule in SL.LABELS if rule in CHOSEN]]
chosen_table

In [ ]:
# 보정 구간에서는 어땠나 (고른 값은 보정 구간에 맞춘 것이라 여기서 더 좋게 나온다. 평가 구간과 차이가 크면 그 조건은 믿기 어렵다)
both = pd.concat([SL.chosen_rows(calib, CHOSEN).assign(구간='보정'), SL.chosen_rows(test, CHOSEN).assign(구간='평가')])
both.pivot(index='룰', columns='구간', values=['고른_거래', '정밀도', '재현율', '하루_평균']).loc[[SL.LABELS[rule] for rule in SL.LABELS if rule in CHOSEN]]

## 7. 무엇을 잡았나
고른 조건에서 거래마다 룰에 걸렸는지를 가져와, 패턴별로 본다. 값은 그 패턴의 세탁 거래 중 그 룰이 고른 비율이다.

In [ ]:
F = SL.collect(con, CANDS, CHOSEN, CALIB_WINDOW[0], TEST_WINDOW[1])
in_test = RL.day_mask(F, TEST_WINDOW)                 # 평가 구간이고 지표에서 빼지 않는 거래
each = {SL.LABELS[rule]: F[SL.LABELS[rule]].to_numpy() for rule in CHOSEN}
any_hit = np.logical_or.reduce(list(each.values()))   # 네 룰 중 하나라도 고른 거래
pattern_table = RL.by_pattern(F, in_test, {**each, '4종 합집합': any_hit})
pattern_table

In [ ]:
# 지표에서 뺀 세탁(무패턴 · STACK · BIPARTITE)은 얼마나 골랐나. 평가 구간. 패턴 룰은 무패턴을 못 잡으므로 일반 룰이 잡는지 따로 본다
day = pd.to_datetime(F.day)
left_out = ((day >= pd.Timestamp(TEST_WINDOW[0])) & (day <= pd.Timestamp(TEST_WINDOW[1])) & F.skip.astype(bool)).to_numpy()
rows = []
for kind in sorted(set(F.pattern_type[left_out])):
    base = left_out & (F.pattern_type == kind).to_numpy()
    rows.append(dict(패턴=kind, 세탁_거래=int(base.sum()), **{name: round(int((mask & base).sum()) / int(base.sum()), 4) for name, mask in {**each, '4종 합집합': any_hit}.items()}))
skip_table = pd.DataFrame(rows)
skip_table

In [ ]:
# 네 룰을 다 켜면 (평가 구간). 재현율 = 6종 패턴 세탁 중 고른 비율
union_table = pd.DataFrame([RL.method_metrics(name, mask, F, in_test) for name, mask in {'4종 합집합': any_hit, **each}.items()])
union_table

## 8. 패턴 룰 · 모델과 겹치는 정도
03번이 저장한 `hits.parquet`(패턴 룰 6종이 고른 거래, 모델이 고른 거래)과 맞춰 본다. 평가 구간.

- `패턴_룰도_고름`: 일반 룰이 고른 거래 중 패턴 룰 6종 중 하나도 고른 거래
- `모델도_고름`: 그중 이진 모델 점수가 의심 기준 이상인 거래 (03번에 모델 점수가 없었으면 이 열은 없다)
- `일반_룰만`: 패턴 룰도 모델도 안 고른 거래. `일반_룰만_세탁` 이 일반 룰이 새로 보태는 몫이다

In [ ]:
HITS = RULE_DIR / 'hits.parquet'
if not HITS.exists():
    overlap_table = None
    print('03번의 hits.parquet 가 없어 건너뛴다')
else:
    hits = pd.read_parquet(HITS)
    pattern_rules = [name for name in RL.RULES.values() if name in hits.columns]
    seen = F[['edge_id']].merge(hits.assign(pattern_rule=hits[pattern_rules].any(axis=1))[['edge_id', 'pattern_rule'] + (['model'] if 'model' in hits else [])],
                                on='edge_id', how='left')
    assert len(seen) == len(F)
    by_rule = seen.pattern_rule.fillna(False).to_numpy(dtype=bool)        # hits.parquet 에 없는 거래 = 패턴 룰도 모델도 안 고른 정상 거래
    by_model = seen.model.fillna(False).to_numpy(dtype=bool) if 'model' in seen else None
    y, rows = F.y.to_numpy(), []
    for name, mask in {**each, '4종 합집합': any_hit}.items():
        chosen = mask & in_test
        alone = chosen & ~by_rule & (~by_model if by_model is not None else True)
        row = dict(룰=name, 고른_거래=int(chosen.sum()), 세탁=int((chosen & (y == 1)).sum()), 패턴_룰도_고름=int((chosen & by_rule).sum()))
        if by_model is not None:
            row['모델도_고름'] = int((chosen & by_model).sum())
        rows.append(dict(**row, 일반_룰만=int(alone.sum()), 일반_룰만_세탁=int((alone & (y == 1)).sum())))
    overlap_table = pd.DataFrame(rows)
overlap_table

## 9. 저장
`results/<DATASET>/scenario_rules/` 에 표와 고른 조건을 남긴다. `run.json` 의 `chosen` 이 템플릿 기본값으로 옮길 값이다.

In [ ]:
calib.to_csv(OUT / 'sweep_calib.csv', index=False)
test.to_csv(OUT / 'sweep_test.csv', index=False)
chosen_table.to_csv(OUT / 'chosen_test.csv')
pattern_table.to_csv(OUT / 'by_pattern.csv', index=False)
skip_table.to_csv(OUT / 'by_excluded.csv', index=False)
union_table.to_csv(OUT / 'union.csv', index=False)
if overlap_table is not None:
    overlap_table.to_csv(OUT / 'overlap.csv', index=False)
F[any_hit | (F.y.to_numpy() == 1)].to_parquet(OUT / 'hits.parquet', index=False)      # 일반 룰이 고른 거래 + 세탁 거래
plain = lambda c: {k: (v.item() if hasattr(v, 'item') else v) for k, v in c.items() if k not in DROP}
json.dump(dict(dataset=DATASET, rule_run=RULE_RUN, test_window=list(TEST_WINDOW), calib_window=list(CALIB_WINDOW), pad_days=PAD_DAYS,
               exclude=EXCLUDE, busy=[BUSY_MAX, BUSY_MINUTES], filters={k: list(v) for k, v in FILTERS.items()}, report_usd=REPORT_USD,
               pick_by=PICK_BY, min_picked=MIN_PICKED, candidates=len(CANDS),
               chosen={rule: plain(CANDS[number]) for rule, number in CHOSEN.items()},
               hypothesis={rule: plain(CANDS[number]) for rule, number in GUESS.items()}, duckdb=duckdb.__version__),
          open(OUT / 'run.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
sorted(p.name for p in OUT.iterdir())